# RAG lab: search this knowledge base

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/rag.ipynb)

Build the retrieval half of a RAG app over this repository's own topic pages: chunking, keyword search, dense search, hybrid search, evaluation, and the final prompt. Companion to [RAG basics](../07-applications/rag.md). Runs on CPU with no API key.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import os
from collections import Counter
from s2a.rag import (Chunk, Retriever, load_markdown_corpus, recall_at_k, build_rag_messages, tokenize,
                     REPO_EVAL_SET, REPO_PARAPHRASE_SET)
from s2a.prompting import render_chatml

# The topic pages: from this checkout if we're in one, otherwise from GitHub (e.g. on Colab).
ROOT = ".." if os.path.exists("../03-foundations") else "silicon-to-agents"
if not os.path.exists(ROOT):
    !git clone -q --depth 1 https://github.com/ankitrgupta1/silicon-to-agents

## 1. Chunk the documents

Split each page under its headings into pieces of at most 150 words, with a little overlap so nothing is lost at a boundary. Each chunk remembers where it came from, so answers can cite it.

In [ ]:
chunks = load_markdown_corpus(ROOT)
print(f"{len(chunks)} chunks from {len({c.source for c in chunks})} pages")
print("largest pages:", Counter(c.source for c in chunks).most_common(3))
example = next(c for c in chunks if c.source.endswith("attention.md") and c.heading == "Intuition")
print("\nexample chunk:", example.source, "›", example.heading)
print(example.text)

## 2. Ask a question three ways

- **Keyword (BM25):** scores chunks by shared words, weighting rare words more.
- **Dense:** turns text into vectors and ranks by cosine similarity. Here the vectors come from LSA (TF-IDF compressed with the SVD from the [linear algebra lab](linear-algebra.ipynb)), a small stand-in for a neural embedding model.
- **Hybrid:** merges the two rankings with reciprocal rank fusion.

In [ ]:
retriever = Retriever(chunks)
question = "how much GPU memory does the KV cache need per token"
for method in ["bm25", "dense", "hybrid"]:
    print(f"--- {method} ---")
    for c in retriever.search(question, k=3, method=method):
        print("  ", c)

## 3. Where keyword search falls short

Keyword search can't match words that aren't there. Dense vectors can, if the words appeared together elsewhere. Here, the first document never says "automobile".

In [ ]:
docs = ["my car engine keeps stalling at red lights", "the automobile and car industry sells millions of cars",
        "an automobile is another word for a car", "car and automobile insurance prices rose this year",
        "the recipe calls for two cups of flour and sugar", "bake the cake for forty minutes until golden",
        "fresh bread needs flour water yeast and salt", "whisk the eggs with sugar before adding flour"]
small = Retriever([Chunk(f"doc{i}", "", t) for i, t in enumerate(docs)], dim=2)
q = tokenize("automobile")
print(f"'{docs[0]}'")
print(f"  keyword score {small.bm25.scores(q)[0]:.2f}, dense score {small.lsa.scores(q)[0]:.2f}")

## 4. Measure, don't guess

Two small question sets about this repo, each with the page that answers it. The second is worded to avoid the pages' key terms. **Recall@k** is the share of questions where a right page appears in the top k results.

In [ ]:
for name, eval_set in [("keyword-style", REPO_EVAL_SET), ("paraphrased", REPO_PARAPHRASE_SET)]:
    print(f"{name} ({len(eval_set)} questions)")
    for method in ["bm25", "dense", "hybrid"]:
        r = [recall_at_k(retriever, eval_set, k, method) for k in (1, 3, 5)]
        print(f"  {method:>6}: recall@1 {r[0]:.2f}  recall@3 {r[1]:.2f}  recall@5 {r[2]:.2f}")

## 5. Build the prompt

The retrieved chunks go into the prompt as numbered sources, with instructions to answer only from them, cite them, and admit when they don't contain the answer. Send this to any chat model.

In [ ]:
question = "Why does decode get slower as more users share a GPU?"
prompt = render_chatml(build_rag_messages(question, retriever.search(question, k=3)))
print(prompt[:1500], "...")

## Try it

- Write five questions of your own about this repo, with the page that answers each. How do the three methods do?
- Change the chunk size in `load_markdown_corpus` (via `chunk_markdown`'s `max_words`) to 50 and 400. What happens to recall@1?
- Find a question where keyword search beats dense search, and one where the reverse is true.
- Ask about something the repo doesn't cover ("what is speculative decoding?"). What does retrieval return, and why does the prompt's "say you don't know" rule matter?